In [27]:
import asyncio, re, textwrap
from urllib.parse import urlparse, parse_qs

import aiohttp                # async HTTP client
import requests               # sync HTTP client (for the search page)
from bs4 import BeautifulSoup # HTML parser

In [28]:
GOOGLE_SEARCH_URL = "https://www.google.com/search"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/605.1.15 (KHTML, like Gecko) Version/18.5 Safari/605.1.15"
    )
}

### do search and collect 3 result links

In [29]:
def bing_search(query: str, max_links: int = 3) -> list[str]:
    url = "https://www.bing.com/search"
    html = requests.get(url, params={"q": query}, headers=HEADERS, timeout=15).text
    soup = BeautifulSoup(html, "html.parser")

    links = []
    for a in soup.select("li.b_algo h2 a"):      # Bing’s organic result selector
        links.append(a["href"])
        if len(links) == max_links:
            break
    return links


### keep plain text

In [30]:
def clean_html(html: str) -> str:
    """drop scripts, styles, nav, footers, etc."""
    soup = BeautifulSoup(html, "html.parser")

    # Remove obvious noise
    for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside"]):
        tag.decompose()

    # Collect paragraph text
    paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
    text = "\n".join(p for p in paragraphs if p)

    # Optionally wrap long lines for nicer viewing/debugging
    return textwrap.fill(text, width=100)

### fetch 3 pages concurrently

### threads

In [31]:
from concurrent.futures import ThreadPoolExecutor
import requests

def fetch_sync(url):
    return clean_html(requests.get(url, headers=HEADERS, timeout=20).text)

def fetch_all_sync(urls):
    with ThreadPoolExecutor(max_workers=len(urls)) as pool:
        return list(pool.map(fetch_sync, urls))


#### run all together

In [42]:
def scrape_query(query: str) -> str:
    """search → fetch top 3 pages → return concatenated plain text."""
    links = google_search(query, 3) or bing_search(query, 3)

    if not links:
        raise RuntimeError("No links found search blocked or layout changed.")

    """print("[i] Found links:")
    for i, link in enumerate(links, 1):
        print(f"  {i}. {link}")"""

    texts = fetch_all_sync(links)
    combined = "\n".join(texts)
    return combined

In [43]:
if __name__ == "__main__":
    query = "Stevens Institute of Technology professor Tian" 
    result_text = scrape_query(query)
    print(result_text[:3000], "...") 

Quantum, AI, national defense, health, extreme-weather resilience and critical minerals are all
strengths of Stevens’ robust — and growing — research enterprise Ready to shape tomorrow? At Stevens
Institute of Technology, we're looking for innovators, dreamers and problem-solvers like you.
Located minutes from NYC, Stevens combines cutting-edge technology with collaborative spirit to
prepare the next generation of leaders.  Our students don't just imagine the future — they create
it. From breakthrough research to entrepreneurial ventures, every Stevens student has the
opportunity to make their mark. With exceptional career outcomes and a supportive community, you'll
find everything you need to turn your passion into impact. Technology is at the core of every one of
Stevens’ dozens of degree programs, so no matter what you study, you’ll be at the leading edge of
the industry. Discover your passion, then turn it into a career that’s personally and professionally
rewarding.
Stevens Instit